# 02 — Data Preprocessing

This notebook cleans the raw dataset and prepares it for feature engineering.

**Steps:**
1. Remove PII and identifier columns
2. Remove data leakage columns
3. Handle missing values
4. Extract date features
5. Save cleaned data

In [1]:
import pandas as pd
import numpy as np
import sys
import warnings

warnings.filterwarnings('ignore')

# Add project root to path for imports
sys.path.insert(0, '..')
from src.preprocessing import load_data, clean_data, remove_leakage_columns, get_column_summary

## 2.1 Load Raw Data

In [2]:
df = load_data("../data/raw/DataCoSupplyChainDataset.csv")
df.head(3)

Loaded dataset: 180519 rows, 53 columns


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class


## 2.2 Remove Data Leakage Columns

**This is critical.** Three columns contain information that would only be available
AFTER delivery has happened. Using them to predict delivery risk would be cheating.

| Column | Why it leaks |
|--------|-------------|
| `Days for shipping (real)` | Actual shipping duration — only known after delivery |
| `Delivery Status` | Final delivery outcome — this IS what we're predicting |
| `shipping date (DateOrders)` | When the item actually shipped — not known at order time |

In [3]:
df = remove_leakage_columns(df)
print(f"Shape after leakage removal: {df.shape}")

Removed leakage columns: ['Days for shipping (real)', 'Delivery Status', 'shipping date (DateOrders)']
Shape after leakage removal: (180519, 50)


## 2.3 Clean the Data

The `clean_data()` function handles:
- Removing PII columns (email, names, password, street)
- Removing identifiers (customer ID, order ID, etc.)
- Removing lat/lng (too granular, and we already have city/region)
- Removing `Order Zipcode` (86% missing — not useful)
- Removing `Product Description` (100% missing)
- Dropping duplicates
- Extracting date features (Year, Month, Day, DayOfWeek, Hour)

In [4]:
df = clean_data(df)

Dropped 18 PII/ID/unusable columns


Extracted date features: Year, Month, Day, DayOfWeek, Hour
Cleaning complete: (180519, 50) → (180519, 37)


In [5]:
df.head(3)

,Type,Days for shipment (scheduled),Benefit per order,Sales per customer,Late_delivery_risk,Category Name,Customer City,Customer Country,Customer Segment,Customer State,...,Order Status,Product Name,Product Price,Product Status,Shipping Mode,Order_Year,Order_Month,Order_Day,Order_DayOfWeek,Order_Hour
0,DEBIT,4,91.250000,314.640015,0,Sporting Goods,Caguas,Puerto Rico,Consumer,PR,...,COMPLETE,Smart watch,327.75,0,Standard Class,2018,1,31,Wednesday,22
1,TRANSFER,4,-249.089996,311.359985,1,Sporting Goods,Caguas,Puerto Rico,Consumer,PR,...,PENDING,Smart watch,327.75,0,Standard Class,2018,1,13,Saturday,12
2,CASH,4,-247.779999,309.720001,0,Sporting Goods,San Jose,EE. UU.,Consumer,CA,...,CLOSED,Smart watch,327.75,0,Standard Class,2018,1,13,Saturday,12


## 2.4 Verify Remaining Columns

In [6]:
summary = get_column_summary(df)
summary

,dtype,non_null,missing,missing_pct,n_unique
Type,object,180519,0,0.0,4
Days for shipment (scheduled),int64,180519,0,0.0,4
Benefit per order,float64,180519,0,0.0,21998
Sales per customer,float64,180519,0,0.0,2927
Late_delivery_risk,int64,180519,0,0.0,2
Category Name,object,180519,0,0.0,50
Customer City,object,180519,0,0.0,563
Customer Country,object,180519,0,0.0,2
Customer Segment,object,180519,0,0.0,3
Customer State,object,180519,0,0.0,46


In [7]:
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
num_cols = df.select_dtypes(exclude=["object"]).columns.tolist()

print(f"Categorical columns: {len(cat_cols)}")
print(f"Numerical columns: {len(num_cols)}")
print(f"\nCategorical: {cat_cols}")
print(f"\nNumerical: {num_cols}")

Categorical columns: 16
Numerical columns: 21

Categorical: ['Type', 'Category Name', 'Customer City', 'Customer Country', 'Customer Segment', 'Customer State', 'Department Name', 'Market', 'Order City', 'Order Country', 'Order Region', 'Order State', 'Order Status', 'Product Name', 'Shipping Mode', 'Order_DayOfWeek']

Numerical: ['Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Late_delivery_risk', 'Customer Zipcode', 'order date (DateOrders)', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Product Price', 'Product Status', 'Order_Year', 'Order_Month', 'Order_Day', 'Order_Hour']


## 2.5 Check Remaining Missing Values

In [8]:
remaining_missing = df.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0]

if len(remaining_missing) > 0:
    print("Remaining missing values:")
    print(remaining_missing)
else:
    print("No missing values remaining in core columns")

Remaining missing values:


Customer Zipcode    3
dtype: int64


## 2.6 Save Cleaned Data

Saving the cleaned dataset for the next notebook. We keep the date column
so it can be used for time-series analysis later.

In [9]:
# Drop the original date column now (features already extracted)
if "order date (DateOrders)" in df.columns:
    # Save a copy WITH dates for forecasting notebook
    df.to_csv("../data/processed/cleaned_with_dates.csv", index=False)
    print("Saved cleaned data with dates for forecasting")
    
    # Drop date for the ML pipeline
    df_ml = df.drop(columns=["order date (DateOrders)"])
else:
    df_ml = df.copy()

df_ml.to_csv("../data/processed/cleaned_data.csv", index=False)
print(f"Saved cleaned ML data: {df_ml.shape}")
print(f"\nFinal columns ({len(df_ml.columns)}): {list(df_ml.columns)}")

Saved cleaned data with dates for forecasting


Saved cleaned ML data: (180519, 36)

Final columns (36): ['Type', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Late_delivery_risk', 'Category Name', 'Customer City', 'Customer Country', 'Customer Segment', 'Customer State', 'Customer Zipcode', 'Department Name', 'Market', 'Order City', 'Order Country', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status', 'Product Name', 'Product Price', 'Product Status', 'Shipping Mode', 'Order_Year', 'Order_Month', 'Order_Day', 'Order_DayOfWeek', 'Order_Hour']


## Summary

**What we did:**
- Removed 3 leakage columns that contained post-delivery information
- Removed 18 PII/ID/unusable columns
- Extracted 5 date features from order date
- Saved cleaned data for the next stage

**Important decision:** We kept `Order Profit Per Order` and `Benefit per order`
because these represent the expected/estimated profit at order time, not the
actual realized profit. Similarly, `Sales` represents the order value which
is known when the order is placed.

**Next step:** Feature engineering